# Adatok előfeldolgozása

A [D-PLACE](https://d-place.org/) (Database of Places, Language, Culture, and Environment) egy nyilvános adatbázis, amely a világ kulturális, nyelvi és környezeti sokféleségét dokumentálja. Ezen belül a **Standard Cross-Cultural Sample (SCCS)** Murdock és White (1969) gondosan összeválogatott mintája: 186 társadalmat tartalmaz a világ minden részéről, úgy kiválasztva, hogy minimalizálják az egymásra hatást (kulturális diffúzió, ún. Galton-probléma). Ezáltal az SCCS különösen alkalmas társadalmak közötti összehasonlító vizsgálatokra.

Az adatokat a [D-PLACE GitHub repóból](https://github.com/D-PLACE/dplace-data) töltjük le közvetlenül, CSV formátumban.

In [1]:
import pandas as pd
import numpy as np
import re

## 1. Nyers adatok letöltése

A D-PLACE az SCCS adatokat több CSV fájlra bontja, amelyek együtt alkotják a teljes adatbázist:

| Fájl | Tartalom |
|---|---|
| `data.csv` | A tulajdonképpeni adatok: minden sor egy társadalom-változó pár, a hozzá tartozó kódolt értékkel |
| `societies.csv` | A 186 társadalom metaadatai: név, földrajzi koordináták |
| `variables.csv` | A változók leírásai: cím, kategória, definíció, típus |
| `codes.csv` | A kategorikus változók kódjainak jelentése (pl. mit jelent az 1-es vagy 2-es kód egy adott változónál) |
| `societies_mapping.csv` | Kereszthivatkozás az Ethnographic Atlas-szal - ebből nyerjük ki a földrajzi régiót |

In [2]:
base = 'https://raw.githubusercontent.com/D-PLACE/dplace-data/master/datasets/SCCS/'

data = pd.read_csv(base + 'data.csv', low_memory=False)
societies = pd.read_csv(base + 'societies.csv')
variables = pd.read_csv(base + 'variables.csv')
codes = pd.read_csv(base + 'codes.csv')
mapping = pd.read_csv(base + 'societies_mapping.csv')

print(f'data:      {data.shape[0]:,} rows x {data.shape[1]} cols')
print(f'societies: {societies.shape[0]} rows x {societies.shape[1]} cols')
print(f'variables: {variables.shape[0]:,} rows x {variables.shape[1]} cols')
print(f'codes:     {codes.shape[0]:,} rows x {codes.shape[1]} cols')
print(f'mapping:   {mapping.shape[0]} rows x {mapping.shape[1]} cols')

data:      329,449 rows x 9 cols
societies: 186 rows x 15 cols
variables: 1,781 rows x 9 cols
codes:     10,190 rows x 4 cols
mapping:   186 rows x 2 cols


## 2. Nyers adatok áttekintése

Az egyes táblák szerkezete és néhány példasor:

In [3]:
print('=== data.csv ===')
print('Columns:', data.columns.tolist())
data.head()

=== data.csv ===
Columns: ['soc_id', 'sub_case', 'year', 'var_id', 'code', 'comment', 'references', 'source_coded_data', 'admin_comment']


,soc_id,sub_case,year,var_id,code,comment,references,source_coded_data,admin_comment
0,SCCS1,Gei/Khauan tribe,1860.0,SCCS860,3.0,NaN,NaN,STDS37.DAT WC10.1,Date updated with v875
1,SCCS10,Morogoro District,NaN,SCCS860,3.0,NaN,NaN,STDS37.DAT WC10.1,Date updated with v875
2,SCCS100,Ravenga District,1920.0,SCCS860,4.0,NaN,NaN,STDS37.DAT WC10.1,Date updated with v875
3,SCCS101,Bunlap village,1950.0,SCCS860,4.0,NaN,NaN,STDS37.DAT WC10.1,Date updated with v875
4,SCCS102,"Bau Chiefdom, Vanua Levu",NaN,SCCS860,3.0,NaN,NaN,STDS37.DAT WC10.1,Date updated with v875


In [4]:
print('=== societies.csv ===')
print('Columns:', societies.columns.tolist())
societies.head()

=== societies.csv ===
Columns: ['id', 'xd_id', 'pref_name_for_society', 'glottocode', 'ORIG_name_and_ID_in_this_dataset', 'alt_names_by_society', 'main_focal_year', 'HRAF_name_ID', 'HRAF_link', 'origLat', 'origLong', 'Lat', 'Long', 'Comment', 'glottocode_comment']


,id,xd_id,pref_name_for_society,glottocode,ORIG_name_and_ID_in_this_dataset,alt_names_by_society,main_focal_year,HRAF_name_ID,HRAF_link,origLat,origLong,Lat,Long,Comment,glottocode_comment
0,SCCS2,xd1,!Kung,juho1239,Kung (SCCS2),"Kung Bushmen, !Kung (Was Nyae), Was Nyae",1950,San (FX10),http://ehrafworldcultures.yale.edu/collection?...,-19.83,20.58,-19.83,20.58,Original,NaN
1,SCCS1,xd3,Nama,nama1265,Nama (SCCS1),"Khoekhoe, Namaqua, Nama Hottentot",1860,Khoi (FX13),http://ehrafworldcultures.yale.edu/collection?...,-27.50,17.00,-27.50,17.00,Original,NaN
2,SCCS13,xd5,Mbuti,bila1255,Mbuti (SCCS13),"Sua, BaSua, Kango, BaKango, Bambuti, BaMbuti P...",1950,Mbuti (FO04),http://ehrafworldcultures.yale.edu/collection?...,1.50,28.33,1.50,28.33,Original,"Note 8Feb2018: Changed. Focus is ""Epulu net-hu..."
3,SCCS9,xd9,Hadza,hadz1240,Hadza (SCCS9),"Hadzabe, Hatsa Kindiga, Hadza, Kindiga",1930,NaN,NaN,-3.75,35.18,-3.75,35.18,Original,NaN
4,SCCS4,xd26,Lozi,lozi1239,Lozi (SCCS4),NaN,1900,Lozi (FQ09),http://ehrafworldcultures.yale.edu/collection?...,-16.00,23.50,-16.00,23.50,Original,NaN


In [5]:
print('=== variables.csv ===')
print('Columns:', variables.columns.tolist())
variables.head()

=== variables.csv ===
Columns: ['id', 'category', 'title', 'definition', 'type', 'units', 'source', 'changes', 'notes']


,id,category,title,definition,type,units,source,changes,notes
0,SCCS1,"Subsistence, Economy",Intercommunity Trade as Food Source,"Murdock, G. P., & Morrow, D. O. (1970). Subsis...",Ordinal,NaN,murdock1970subsistence,NaN,NaN
1,SCCS2,"Subsistence, Economy",Food Import Acquisition,"Murdock, G. P., & Morrow, D. O. (1970). Subsis...",Categorical,NaN,murdock1970subsistence,NaN,NaN
2,SCCS3,"Subsistence, Economy",Agriculture - Contribution to Local Food Supply,"Murdock, G. P., & Morrow, D. O. (1970). Subsis...",Categorical,NaN,murdock1970subsistence,NaN,NaN
3,SCCS4,"Subsistence, Economy",Crops - Principal,"Murdock, G. P., & Morrow, D. O. (1970). Subsis...",Categorical,NaN,murdock1970subsistence,NaN,NaN
4,SCCS5,"Subsistence, Economy",Animal Husbandry - Contribution to Food Supply,"Murdock, G. P., & Morrow, D. O. (1970). Subsis...",Categorical,NaN,murdock1970subsistence,NaN,NaN


## 3. Változók kiválasztása

Az SCCS adatbázis közel 2000 változót tartalmaz. A vizsgálatunkhoz az alábbi ötöt választjuk ki:

- **SCCS588 - Community-wide Exclusively Female Work Groups** (függő változó): Léteznek-e az adott társadalomban olyan munkacsoportok, amelyek kizárólag nőkből állnak? Ez egy ordinális kategorikus változó (1 = nincs, 2 = van egy vagy több tevékenységre).

- **SCCS826 - Average Female Contribution to Subsistence** (fő független változó): A nők átlagos hozzájárulása a megélhetéshez, százalékban kifejezve. Ez egy folytonos változó (0-100%).

- **SCCS3 - Agriculture: Contribution to Local Food Supply** (kontroll): A mezőgazdaság hozzájárulása a helyi élelmiszerellátáshoz (1-6 skála, a "nincs"-től a "döntően mezőgazdasági"-ig).

- **SCCS69 - Marital Residence** (kontroll): Házassági lakóhely - hol él az új házaspár? (1 = matrilokális/a feleség családjánál, 3 = patrilokális/a férj családjánál, stb.)

- **SCCS64 - Population Density** (kontroll): Népsűrűség hétfokú skálán (1 = nagyon ritka, 7 = nagyon sűrű).

In [6]:
target_vars = ['SCCS588', 'SCCS826', 'SCCS3', 'SCCS69', 'SCCS64']

variables = variables.rename(columns={'id': 'var_id'})
var_info = variables[variables['var_id'].isin(target_vars)][['var_id', 'title', 'category', 'type']]
var_info

,var_id,title,category,type
2,SCCS3,Agriculture - Contribution to Local Food Supply,"Subsistence, Economy",Categorical
63,SCCS64,Population Density (from Murdock and Wilson Data),"Settlement, Community",Categorical
68,SCCS69,Marital Residence,"Settlement, Community",Categorical
564,SCCS588,Community-wide Exclusively Female Work Groups,Gender,Ordinal
801,SCCS826,Average Female Contribution to Subsistence,"Gender, Data Quality",Continuous


## 4. Kódértékek

A kategorikus változóknál a `codes.csv` tartalmazza, hogy az egyes számkódok mit jelentenek. Ez fontos az eredmények értelmezéséhez - például az SCCS69-nél az 1-es kód matrilokális lakóhelyet jelent, a 3-as patrilokálisat.

In [7]:
for vid in target_vars:
    title = var_info[var_info['var_id'] == vid]['title'].values[0]
    print(f'=== {vid}: {title} ===')
    c = codes[codes['var_id'] == vid][['code', 'description', 'name']].copy()
    print(c.to_string(index=False))
    print()

=== SCCS588: Community-wide Exclusively Female Work Groups ===
 code              description                     name
  NaN             Missing data                      NaN
  1.0                      NaN                      NaN
  2.0 For one or more activity For one or more activity

=== SCCS826: Average Female Contribution to Subsistence ===
 code  description name
  NaN Missing data  NaN

=== SCCS3: Agriculture - Contribution to Local Food Supply ===
 code                                                   description                   name
  NaN                                                  Missing data                    NaN
  1.0                                                           NaN                    NaN
  2.0                                                Non-food Crops         Non-Food Crops
  3.0                                                      < 10 pct                  < 10%
  4.0 < 50 pct , and less than any other single source, incl. trade  < 50% < single s

## 5. Szűrés

A `data.csv` az összes SCCS változó értékét tartalmazza (329 ezer sor). Ebből csak az 5 kiválasztott változó sorait tartjuk meg, és a többi táblából is csak a szükséges oszlopokat.

In [8]:
data = data[['soc_id', 'var_id', 'code']].copy()
data = data[data['var_id'].isin(target_vars)].copy()
print(f'Filtered data: {data.shape[0]} rows')

societies = societies.rename(columns={'id': 'soc_id'})
societies = societies[['soc_id', 'pref_name_for_society', 'Lat', 'Long']].copy()

variables = variables[['var_id', 'title', 'category']].copy()

Filtered data: 930 rows


## 6. Földrajzi régiók hozzárendelése

A regionális elemzéshez szükségünk van a társadalmak világrégiójára. A D-PLACE a `societies_mapping.csv` fájlban minden SCCS társadalomhoz megadja az Ethnographic Atlas (EA) azonosítóját. Ez az azonosító egy kódot tartalmaz (pl. `[Aa3]`, `[Sc6]`), amelynek első betűje a Murdock-féle mintavételi régiót jelöli:

| Betű | Régió | Példa |
|---|---|---|
| A | Africa | `[Aa3]` - Nama Hottentot |
| C | Circum-Mediterranean | `[Cj3]` - Egyptians |
| E | East Eurasia | `[Eh4]` - Vietnamese |
| I | Insular Pacific | `[Ie5]` - Tikopia |
| N | North America | `[Na5]` - Klamath |
| S | South America | `[Sc6]` - Saramacca |

Ez a hat régió az SCCS eredeti mintavételi kerete - a társadalmakat úgy választották ki, hogy minden régió képviselve legyen.

In [9]:
print('societies_mapping.csv:')
print(mapping.head(5).to_string())

societies_mapping.csv:
        id                       related
0  SCCS164  EA: Barama Carib (Sc3) [Sc3]
1    SCCS8      EA: Nyakyusa (Ad6) [Ad6]
2  SCCS166     EA: Mundurucu (Sd1) [Sd1]
3  SCCS167         EA: Cubeo (Se5) [Se5]
4  SCCS160      EA: Haitians (Sb9) [Sb9]


In [10]:
def extract_region(related):
    m = re.search(r'\[([A-Z])[a-z]\d', str(related))
    if m:
        prefix = m.group(1)
        return {'A': 'Africa', 'C': 'Circum-Mediterranean', 'E': 'East Eurasia',
                'I': 'Insular Pacific', 'N': 'North America', 'S': 'South America'}.get(prefix)
    return None

mapping = mapping.rename(columns={'id': 'soc_id'})
mapping['region'] = mapping['related'].apply(extract_region)

print('Region distribution:')
print(mapping['region'].value_counts().to_string())
print(f'\nMissing: {mapping["region"].isna().sum()}')

Region distribution:
region
East Eurasia            34
North America           33
South America           32
Insular Pacific         31
Africa                  28
Circum-Mediterranean    28

Missing: 0


In [11]:
societies = pd.merge(societies, mapping[['soc_id', 'region']], on='soc_id', how='left')
societies.head()

,soc_id,pref_name_for_society,Lat,Long,region
0,SCCS2,!Kung,-19.83,20.58,Africa
1,SCCS1,Nama,-27.50,17.00,Africa
2,SCCS13,Mbuti,1.50,28.33,Africa
3,SCCS9,Hadza,-3.75,35.18,Africa
4,SCCS4,Lozi,-16.00,23.50,Africa


## 7. Összekapcsolás

A három táblát (szűrt adatok, változó-leírások, társadalom-metaadatok) egyesítjük egyetlen **long format** táblába, ahol minden sor egy társadalom-változó pár az összes hozzá tartozó információval.

In [12]:
d_long = pd.merge(data, variables, on='var_id', how='left')
d_long = pd.merge(d_long, societies, on='soc_id', how='left')
d_long = d_long.sort_values(['soc_id', 'var_id']).reset_index(drop=True)

print(f'd_long: {d_long.shape[0]} rows x {d_long.shape[1]} cols')
print(f'Columns: {d_long.columns.tolist()}')
d_long.head(10)

d_long: 930 rows x 9 cols
Columns: ['soc_id', 'var_id', 'code', 'title', 'category', 'pref_name_for_society', 'Lat', 'Long', 'region']


,soc_id,var_id,code,title,category,pref_name_for_society,Lat,Long,region
0,SCCS1,SCCS3,1.0,Agriculture - Contribution to Local Food Supply,"Subsistence, Economy",Nama,-27.50,17.00,Africa
1,SCCS1,SCCS588,NaN,Community-wide Exclusively Female Work Groups,Gender,Nama,-27.50,17.00,Africa
2,SCCS1,SCCS64,1.0,Population Density (from Murdock and Wilson Data),"Settlement, Community",Nama,-27.50,17.00,Africa
3,SCCS1,SCCS69,3.0,Marital Residence,"Settlement, Community",Nama,-27.50,17.00,Africa
4,SCCS1,SCCS826,26.0,Average Female Contribution to Subsistence,"Gender, Data Quality",Nama,-27.50,17.00,Africa
5,SCCS10,SCCS3,6.0,Agriculture - Contribution to Local Food Supply,"Subsistence, Economy",Luguru,-6.83,37.67,Africa
6,SCCS10,SCCS588,NaN,Community-wide Exclusively Female Work Groups,Gender,Luguru,-6.83,37.67,Africa
7,SCCS10,SCCS64,7.0,Population Density (from Murdock and Wilson Data),"Settlement, Community",Luguru,-6.83,37.67,Africa
8,SCCS10,SCCS69,1.0,Marital Residence,"Settlement, Community",Luguru,-6.83,37.67,Africa
9,SCCS10,SCCS826,40.0,Average Female Contribution to Subsistence,"Gender, Data Quality",Luguru,-6.83,37.67,Africa


## 8. Wide format

Az elemzéshez **wide format** táblára is szükségünk van, ahol minden sor egy társadalom, és az oszlopok a különböző változó-értékek. Ezt a long format pivot-olásával állítjuk elő.

In [13]:
d_wide = d_long.pivot(index='soc_id', columns='title', values='code').reset_index()

soc_info = d_long[['soc_id', 'pref_name_for_society', 'Lat', 'Long', 'region']].drop_duplicates('soc_id')
d_wide = pd.merge(soc_info, d_wide, on='soc_id', how='left')

print(f'd_wide: {d_wide.shape[0]} rows x {d_wide.shape[1]} cols')
d_wide.head(10)

d_wide: 186 rows x 10 cols


,soc_id,pref_name_for_society,Lat,Long,region,Agriculture - Contribution to Local Food Supply,Average Female Contribution to Subsistence,Community-wide Exclusively Female Work Groups,Marital Residence,Population Density (from Murdock and Wilson Data)
0,SCCS1,Nama,-27.50,17.00,Africa,1.0,26.0,NaN,3.0,1.0
1,SCCS10,Luguru,-6.83,37.67,Africa,6.0,40.0,NaN,1.0,7.0
2,SCCS100,Tikopia,-12.29,168.83,Insular Pacific,6.0,30.0,NaN,3.0,6.0
3,SCCS101,Bunlap,-15.95,168.22,Insular Pacific,6.0,NaN,1.0,3.0,4.0
4,SCCS102,Mbau Fijians,-18.00,178.58,Insular Pacific,4.0,28.0,NaN,3.0,7.0
5,SCCS103,Ajie,-21.33,165.67,Insular Pacific,5.0,47.0,NaN,3.0,3.0
6,SCCS104,Māori,-35.33,174.17,Insular Pacific,5.0,31.0,NaN,3.0,2.0
7,SCCS105,Marquesans,-8.92,-140.17,Insular Pacific,6.0,18.0,1.0,5.0,5.0
8,SCCS106,Upolu Samoans,-13.94,-171.77,Insular Pacific,6.0,33.0,NaN,3.0,6.0
9,SCCS107,Makin,3.38,172.99,Insular Pacific,6.0,41.0,NaN,3.0,6.0


## 9. Hiányzó értékek

Nem minden változóra áll rendelkezésre adat minden társadalomnál. A fő elemzésben csak azok a társadalmak szerepelhetnek, ahol a függő változó (SCCS588) és a fő független változó (SCCS826) egyaránt rendelkezésre áll. A kontrollváltozóknál mindig az adott változón hiányzó esetek kerülnek kizárásra.

In [14]:
print('Missing values per variable:')
for col in d_wide.columns[5:]:
    n = d_wide[col].isna().sum()
    pct = n / len(d_wide) * 100
    print(f'  {col}: {n} ({pct:.1f}%)')

n_both = d_wide[['Average Female Contribution to Subsistence',
                  'Community-wide Exclusively Female Work Groups']].dropna().shape[0]
print(f'\nBoth SCCS826 and SCCS588 available: {n_both} societies (main analysis sample)')

Missing values per variable:
  Agriculture - Contribution to Local Food Supply: 0 (0.0%)
  Average Female Contribution to Subsistence: 3 (1.6%)
  Community-wide Exclusively Female Work Groups: 114 (61.3%)
  Marital Residence: 1 (0.5%)
  Population Density (from Murdock and Wilson Data): 2 (1.1%)

Both SCCS826 and SCCS588 available: 70 societies (main analysis sample)


## 10. Mentés

A feldolgozott adatokat TSV fájlokba mentjük. Ezeket olvassa be a `beszamolo.ipynb` elemző notebook.

In [15]:
d_long.to_csv('d_long.tsv', sep='\t', index=False)
d_wide.to_csv('d_wide.tsv', sep='\t', index=False)

print(f'd_long.tsv: {d_long.shape[0]} rows x {d_long.shape[1]} cols')
print(f'd_wide.tsv: {d_wide.shape[0]} rows x {d_wide.shape[1]} cols')

d_long.tsv: 930 rows x 9 cols
d_wide.tsv: 186 rows x 10 cols
